In [ ]:
import numpy as np
import cv2 as cv
import matplotlib.pyplot as plt


In [ ]:
!wget -nc https://www.dais.unive.it/~bergamasco/teachingfiles/cvstuff/0004.jpg
!wget -nc https://www.dais.unive.it/~bergamasco/teachingfiles/cvstuff/0005.jpg
!wget -nc https://www.dais.unive.it/~bergamasco/teachingfiles/cvstuff/0004.txt
!wget -nc https://www.dais.unive.it/~bergamasco/teachingfiles/cvstuff/0005.txt

In [ ]:

I0 = cv.cvtColor( cv.imread( "0004.jpg"), cv.COLOR_BGR2RGB )
I1 = cv.cvtColor( cv.imread( "0005.jpg"), cv.COLOR_BGR2RGB )

In [ ]:
data0 = np.loadtxt("0004.txt")
K0 = data0[0:3,:]
R0 = data0[3:6,:]
T0 = data0[6,:]
RT0 = np.block( [[R0,np.expand_dims(T0,axis=-1)],[0,0,0,1]] )
RT0 = np.linalg.inv(RT0) # world 2 camera

data1 = np.loadtxt("0005.txt")
K1 = data1[0:3,:]
R1 = data1[3:6,:]
T1 = data1[6,:]
RT1 = np.block( [[R1,np.expand_dims(T1,axis=-1)],[0,0,0,1]] )
RT1 = np.linalg.inv(RT1) # world 2 camera


RT = RT1@np.linalg.inv(RT0)
R = RT[0:3,0:3]
T = RT[0:3,3]
Tx = np.array( [ [0, -T[2], T[1]],[T[2], 0, -T[0]],[-T[1], T[0], 0] ] )


F = np.linalg.inv(K1).T @ Tx @ R @ np.linalg.inv( K0 )
print("F:",F)
print("det(F):", np.linalg.det(F))

In [ ]:

def line_helper( l, W, H ):
    l = np.array(l)
    l = l/l[2]

    if l[0]>l[1]:
        # vertical line
        p0 = np.cross( l, np.array([1,0,0]))
        p1 = np.cross( l, np.array([1,0,-W]))
    else:
        # horizontal line
        p0 = np.cross( l, np.array([0,1,0]))
        p1 = np.cross( l, np.array([0,1,-H]))

    return p0/p0[2], p1/p1[2]


p = np.array( [1200,600,1])
#p = np.array( [1290,1000,1])
#p = np.array( [630,1350,1])

l = F@p
p0,p1 = line_helper( l, I0.shape[1], I0.shape[0] )

fig = plt.figure( figsize=(18,18))
plt.subplot(1,2,1)
plt.imshow( I0 );
plt.scatter( p[0],p[1],c='r' )
plt.subplot(1,2,2)
plt.imshow( I1 );
plt.plot([p0[0], p1[0]], [p0[1], p1[1]],'-r')

